In [2]:
import math

from scipy.io import arff
from operator import index

import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors, KernelDensity
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import scipy.stats
from scipy.stats import expon, skew, norm,gamma, anderson,goodness_of_fit, monte_carlo_test, probplot, skewnorm
from scipy import integrate
from sklearn.metrics import auc
import seaborn as sns
import math

from statsmodels.sandbox.distributions.gof_new import kstest

plt.rcParams['figure.figsize'] = [15, 7]
import warnings
from scipy import stats

warnings.filterwarnings('ignore')

In [8]:
class ParametricMethod:
    def __init__(self,filename,p,logTrue=False,distribution=stats.gamma):
        self.distance = []
        self.fileName = filename
        self.X = 0
        self.y = 0
        self.arr = []
        self.logTrue = logTrue
        self.p = p
        self.tots = []
        self.distribution = distribution

    def generateOutput(self):
        self._readArff()
        for v in range(2,70):
            self._distanceMetric(v)
            self._generateArray()
            if self.logTrue:
                self.arr = np.log(self.arr)
            params = self.distribution.fit(self.arr) # fit params for gamma distribution
            posNeg4 = []
            spaceStep4 = np.linspace(0,.99,30) # threshold from 0 to .99, 30 samples
            for e in spaceStep4:
                newArr = self.arr > self.distribution.ppf(e,params[0],loc=params[1], scale=params[2]) # if arr value is outside threshold add to new array
                posNeg4.append([((self.y[newArr] == 1).sum() / (self.y == 1).sum()), (self.y[newArr] != 1).sum()/ ((self.y != 1).sum())]) # True positive rate, false positive rate

            posNeg4 = np.array(posNeg4)
            arrtest1, arrtest2 = np.split(posNeg4, 2,axis=1) # split the array
            self.tots += [auc(arrtest2, arrtest1)] # return the area under the curve

        self._printResults(self.tots)

    def _distanceMetric(self,n):
        #find the nearestNeighbors
        nn = NearestNeighbors(n_neighbors=n,p=self.p)
        nn.fit(self.X, self.y)
        #return the dist of each and the nearest neighbors
        self.distance, knn = nn.kneighbors(self.X)  # returns N index neighbors including self

    def _readArff(self):
        arff_file = arff.loadarff(f'./{self.fileName}') # import the attribute-relation file format
        df4 = pd.DataFrame(arff_file[0])
        self.X = df4.drop(columns=['outlier','id']).values
        #get outlier values
        self.y = df4['outlier'].values
        le = LabelEncoder()
        #encoded the variables as 0=non-outlier, 1=outlier
        self.y = le.fit_transform(self.y)

    def _generateArray(self):
        self.arr = []
        #returns an array based on the median and max values
        for x in self.distance:  # finds the distance away from that point (index 0)
            self.arr += [np.max(x)]

    def _printResults(self,totalArr):
        newarr = np.nan_to_num(totalArr)
        newarr = list(newarr)
        print(max(newarr),newarr.index(max(newarr))+2,newarr) #print the max values, the k value, and the array

In [10]:
holder = ParametricMethod("semantic/Arrhythmia/Arrhythmia_withoutdupl_norm_46.arff",1,distribution=stats.skewnorm)
holder.generateOutput()

0.7502785293649531 55 [0.6404882221868533, 0.7410671653668628, 0.7373865987585548, 0.72876213592233, 0.7328008117141493, 0.7231020213273913, 0.7416341715740888, 0.7451555785452809, 0.739704360974057, 0.7425592869648256, 0.7383017666719719, 0.7389682476523953, 0.7412163775266591, 0.7395054114276619, 0.7171832723221391, 0.6993474454878243, 0.6983825401878083, 0.7415048543689321, 0.7466874900525228, 0.74719481139583, 0.745832007003024, 0.7461801687092154, 0.7434147700143243, 0.7403211045678816, 0.7407687410472704, 0.7446482572019735, 0.7448273117937291, 0.7441807257679454, 0.7432357154225689, 0.7395253063823013, 0.7461005888906574, 0.7459613242081808, 0.7456430049339486, 0.744270253063823, 0.7435341397421614, 0.7425194970555467, 0.7413456947318161, 0.7413257997771766, 0.7444095177462996, 0.7426388667833839, 0.7416441190514086, 0.7460409040267388, 0.7471450740092312, 0.7439419863122712, 0.7440812509947475, 0.7463592233009709, 0.7448571542256885, 0.7432954002864873, 0.7451157886360018, 0.74